# 1.- Introducción a Pandas

Pandas no es una simple biblioteca de manipulación de tablas; es una capa de abstracción construida sobre **NumPy** diseñada para el procesamiento vectorizado y el análisis exploratorio de datos. Para trabajar de forma eficiente en entornos de Inteligencia Artificial y Big Data, el primer paso es abandonar la programación procedimental basada en bucles (`for`, `while`) y adoptar el **paradigma vectorizado y declarativo**.

## 1. El modelo mental: de la iteración a la vectorización

En Python estándar, transformar una lista implica recorrer elemento a elemento. En Pandas, las operaciones se ejecutan sobre matrices contiguas en memoria escritas en C. Esto quiere decir que, cuando yo quiero realizar una función sobre todos los elementos de una lista, en Python tradicional tengo que iterar sobre cada elemento de la lista y aplicar sobre dicho elemento la operación deseada. En cambio, en Pandas realizo la operación directamente sobre la lista (en realidad, la estructura de datos de Pandas que equivale a una lista es la serie) y ya se encarga Pandas internamente de aplicar la operación sobre cada elemento de una forma mucho más rápida y eficiente.

En el siguiente código puedes ver un ejemplo de cómo calcular los cuadrados de todos los números de una lista en Python y usando Pandas.

In [4]:
import pandas as pd   # Para usar la librería Pandas primero debemos importarla, por costumbre se renombra como pd

N = 1_000_000   # Recuerda que en Python se puede usar el símbolo _ como separador de miles para aumentar la legibilidad

# Creamos las dos estructuras que contienen los números del 1 al 1000000
datos_list = list(range(N))
serie_pd = pd.Series(range(N))

In [5]:
# Cálculo de los cuadrados en Python
%timeit [x**2 for x in datos_list]

31.8 ms ± 513 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
# Cálculo de los cuadrados en Pandas
%timeit serie_pd ** 2

982 µs ± 67.3 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


En los ejemplos anteriores he usado `%timeit`, un **comando mágico** de Jupyter. Los comandos mágicos permiten realizar operaciones fuera de Python en un cuaderno de Jupyter, por ejemplo, cargar un fichero en una celda o mostrar el directorio de trabajo actual. Si ejecutas `%magic` en una celda obtendrás la ayuda completa de todos estos comandos, mientras que `%quickref` mostrará una referencia rápida de todos estos comandos.

En concreto, el comando `%timeit` sirve para medir el tiempo que tarda en ejecutarse un código determinado. Para ello, lo ejecuta múltiples veces y nos devuelve la media de estas mediciones.

Como podemos ver en la salida del comando anterior, el cálculo en Python tarda unos 32 milisegundos, mientras que la misma operación en Pandas se puede realizar en apenas 800 microsegundos, apenas un 2.5% del tiempo que llevó en Python

## 2. Las dos estructuras de datos fundamentales de Pandas

En Pandas se trabaja con dos tipos de estructuras de datos:

- **Series**: 
- **DataFrames**: una tabla compuesta por un diccionario ordenado de series que comparten un mismo índice de filas

### Series

Una serie es una columna unidimensional con tipado homogéneo y un índice explícito asociado a cada elemento

In [7]:
edades = pd.Series([23, 45, 31, 19], index=['usr_1', 'usr_2', 'usr_3', 'usr_4'], name='edad')

edades

usr_1    23
usr_2    45
usr_3    31
usr_4    19
Name: edad, dtype: int64

En el ejemplo anterior hemos creado una serie con 4 elementos cuyos índices son `usr_1`, `usr_2`, ... Si no hubiéramos indicado estos índices explícitamente Pandas hubiera asignado números secuencialmente. Observa también que Pandas ha inferido el tipo de datos y le ha asignado `int64`.

### DataFrames

Un DataFrame es una tabla compuesta por un diccionario ordenado de series que comparte un mismo índice de filas.

In [8]:
datos = {
    'edad': [23, 45, 31, 19],
    'compras': [120.5, 340.0, 50.2, 890.1],
    'segmento': ['A', 'B', 'B', 'A']
}

df = pd.DataFrame(datos, index=['usr_1', 'usr_2', 'usr_3', 'usr_4'])
df

,edad,compras,segmento
usr_1,23,120.5,A
usr_2,45,340.0,B
usr_3,31,50.2,B
usr_4,19,890.1,A


Algunas funciones y atributos básicos de los DataFrames son:

- `df.shape`: muestra el número de filas y columnas del DataFrame
- `df.dtypes`: tipos de datos por columna
- `df.info()`: resumen de memoria y nulos
- `df.describe()`: métricas estadísticas básicas

In [9]:
df.shape

(4, 3)

In [10]:
df.dtypes

edad          int64
compras     float64
segmento     object
dtype: object

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, usr_1 to usr_4
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   edad      4 non-null      int64  
 1   compras   4 non-null      float64
 2   segmento  4 non-null      object 
dtypes: float64(1), int64(1), object(1)
memory usage: 128.0+ bytes


In [12]:
df.describe()

,edad,compras
count,4.00000,4.000000
mean,29.50000,350.200000
std,11.47461,380.507617
min,19.00000,50.200000
25%,22.00000,102.925000
50%,27.00000,230.250000
75%,34.50000,477.525000
max,45.00000,890.100000


## 3. Acceso, selección y el problema de las copias

El acceso a datos en Pandas suele ser la primera fuente de errores silenciosos y degradación de rendimiento. Comprender los mecanismos internos de indexación y cómo interactúan con la memoria evita comportamientos no deterministas en producción.

## 3.1. Indexación estándar (`[]`) frente a indexadores explícitos

En Python puro, los corchetes `[]` operan siempre sobre la misma dimensión. En un DataFrame, que es una estructura bidimensional, los corchetes pueden ser en ocasiones ambiguos o llevar a errores. Observa en los siguientes ejemplos cómo cambia el resultado según la forma de ponerlos.

In [13]:
df = pd.DataFrame({
    'edad': [25, 42, 37, 19],
    'compras': [120.0, 540.5, 210.2, 85.0],
    'ciudad': ['León', 'Oviedo', 'León', 'Burgos']
}, index=['usr_1', 'usr_2', 'usr_3', 'usr_4'])

# ACCESO DIRECTO df[]: Comportamiento heterogéneo
df['edad']             # Selecciona una COLUMNA (devuelve una Series)
df[['edad', 'ciudad']] # Selecciona varias COLUMNAS (devuelve un DataFrame)
df[0:2]                # Corta FILAS por posición (slice, devuelve un subconjunto de las filas del DataFrame)
df[df['edad'] > 30]    # Filtra FILAS por máscara booleana (devuelve un subconjunto de las filas, que son las que cumplen la condición)

,edad,compras,ciudad
usr_2,42,540.5,Oviedo
usr_3,37,210.2,León


Como puedes ver, usar corchetes puede inducir a error, por lo que deberías reservarlos los corchetes simples `df[...]` **exclusivamente** para proyectar columnas por su nombre (`df['col']` o `df[['a', 'b']]`). Para cualquier selección bidimensional o indexada, utiliza siempre `.loc` o `.iloc`.

Tanto `.loc` como `.iloc` permiten acceder a los elementos de un DataFrame utilizando corchetes, la diferencia entre ambos radica en que `.loc` está basado en las etiquetas de los índices mientras que `.iloc`se basa en la posición. Esto, además, implica un sútil cambio en el comportamiento, ya que cuando usamos `.loc` se incluye el límite superior, pero si usamos `.iloc` el límite superior no se incluye.

Observa el siguiente código:

In [14]:
df

,edad,compras,ciudad
usr_1,25,120.0,León
usr_2,42,540.5,Oviedo
usr_3,37,210.2,León
usr_4,19,85.0,Burgos


In [15]:
# Filtramos las filas desde usr_1 a usr_3 y las columnas edad y compras
df.loc['usr_1':'usr_3', ['edad', 'compras']]

,edad,compras
usr_1,25,120.0
usr_2,42,540.5
usr_3,37,210.2


In [16]:
# Filtramos todas las filas (observa la sintaxis con :) y el rango de columnas 'compras' hasta 'ciudad', ambas incluidas
df.loc[:, 'compras':'ciudad']

,compras,ciudad
usr_1,120.0,León
usr_2,540.5,Oviedo
usr_3,210.2,León
usr_4,85.0,Burgos


In [17]:
# Las dos primeras filas y las dos primeras columnas (observa que la tercera, con índice 2, es excluida)
df.iloc[0:2, 0:2]

,edad,compras
usr_1,25,120.0
usr_2,42,540.5


In [18]:
# Usamos un rango negativo para indicar la última fila y todas las columnas
df.iloc[-1, :]

edad           19
compras      85.0
ciudad     Burgos
Name: usr_4, dtype: object

### 3.2. Acceso a escalares con `.at` y `.iat`

Cuando en un pipeline necesitas extraer o mutar **un único valor escalar**, `.loc` e `.iloc` introducen una sobrecarga innecesaria porque comprueban si el argumento es un slice, una lista o una máscara booleana. En esos casos es más útil utilizar `.at` (etiqueta) y `.iat` (posición), que es entre 5 y 10 veces más rápido para lecturas individuales.

In [19]:
# Acceso al elemento que está en la fila 'usr_1' y columna 'edad'
df.at['usr_1', 'edad']

25

In [20]:
# Acceso al elemento en la primera fila y columna
df.iat[0, 0]

25

## 3.3. Filtrado booleano y máscaras lógicas

El filtrado en Pandas se basa en construir una serie de booleanos que actúa como **máscara** sobre el eje de las filas.

Observa bien la sintaxis para construir la máscara, proyectamos una única columna del DataFrame y le aplicamos el operador de comparación sobre todo el DataFrame completo. Internamente, Pandas realizará esa comparación sobre cada uno de los elementos del DataFrame y generará otro DataFrame con los resultados de estas operaciones.

In [21]:
# Creamos la máscara
mascara = df['edad'] >= 30
print(mascara)

# Aplicación con .loc
df_adultos = df.loc[mascara, ['ciudad', 'compras']]
df_adultos

usr_1    False
usr_2     True
usr_3     True
usr_4    False
Name: edad, dtype: bool


,ciudad,compras
usr_2,Oviedo,540.5
usr_3,León,210.2


In [22]:
df[mascara]

,edad,compras,ciudad
usr_2,42,540.5,Oviedo
usr_3,37,210.2,León


Además de los operadores de comparación, podemos utilizar algunos métodos disponibles en Pandas para generar la máscara:

- `df['ciudad'].isin(['León', 'Burgos'])`: comprueba pertenencia a un conjunto.
- `df['compras'].between(100.0, 300.0)`: rango continuo inclusivo ($[a, b]$).
- `df['ciudad'].isna() / notna()`: comprobación de valores nulos.

### El problema de `SettingWithCopyWarning`

Un problema muy habitual es cuando encadenamos índices de la siguiente forma: `df[df['edad'] > 30]['compras'] = 0.0`. Aunque intuitivamente puede parecer que la sentencia anterior es correcta, si intentamos ejecutarla nos lanzará el error `SettingWithCopyWarning`. La clave para entender esto está en entender qué hace Python con la memoria del ordenador detrás de cada corchete.

Cuando le pides a Pandas un subconjunto de filas o columnas de un DataFrame, pueden ocurrir dos cosas a nivel de memoria:
- Una **vista**): Pandas te da una "lupa" que apunta directamente a las posiciones de memoria del DataFrame original. Si pintas a través de la lupa, modificas el original.
- Una **copia**: Pandas reserva un bloque de memoria nuevo en la RAM y clona los datos. Si pintas sobre la copia, el original queda intacto.

El problema es que Pandas no siempre te avisa de antemano si un filtro generará una vista o una copia (depende de cómo estén distribuidos los tipos de datos en los bloques contiguos de NumPy), es algo que no controlamos.

Veamos por qué ocurre esto. Supón que ejecutamos lo siguiente:

```python
df[df['edad'] > 30]['compras'] = 0.0
```

El intérprete Python lo descompondrá de la siguiente forma:

```python
paso_1 = df[df['edad'] > 30]  # No sabemos si el paso_1 es una vista o una copia
paso_1['compras'] = 0.0       # Modificamos paso_1
```

Aquí surge el conflicto:

- Si `paso_1` resultó ser una copia, acabas de modificar un objeto temporal en la memoria RAM.
- Al terminar esa línea, `paso_1` no está asignado a ninguna variable persistente, por lo que el recolector de basura (Garbage Collector) de Python lo destruye inmediatamente.
- El DataFrame original `df` nunca se enteró de la modificación.
- Pandas detecta esta cadena de operaciones y lanza `SettingWithCopyWarning`: "Ojo, intentaste asignar un valor sobre algo que probablemente era una copia temporal y tu cambio se ha perdido".

Para modificar datos sin riesgo, hay que fusionar la condición de filas y la columna objetivo en **un único par de corchetes**:

```python
df.loc[df['edad'] > 30, 'compras'] = 0.0
```